In [11]:
# 1. Load the CSVs
weather_df = pd.read_csv('../backend/data/historical_weather_magdeburg.csv')
occ_df = pd.read_csv('../backend/data/synthetic_occupancy_data.csv')

# Force identical time formats
weather_df['time'] = pd.to_datetime(weather_df['time'], format='mixed').dt.tz_localize(None).dt.floor('h')
occ_df['time'] = pd.to_datetime(occ_df['time'], format='mixed').dt.tz_localize(None).dt.floor('h')

# 2. Merge
df = pd.merge(weather_df, occ_df, on='time', how='inner')

# 3. Feature Engineering (Crucial for XGBoost)
df['hour'] = df['time'].dt.hour
df['day_of_week'] = df['time'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

print(f"Data ready! Shape: {df.shape}")
df[['time', 'hour', 'temperature_2m', 'occ_kitchen']].head()

Data ready! Shape: (2161, 11)


,time,hour,temperature_2m,occ_kitchen
0,2026-02-09 15:00:00,15,0.9,0
1,2026-02-09 16:00:00,16,1.0,0
2,2026-02-09 17:00:00,17,1.2,0
3,2026-02-09 18:00:00,18,1.0,1
4,2026-02-09 19:00:00,19,0.9,1


In [12]:
# Define our inputs (Features)
features = ['hour', 'day_of_week', 'is_weekend', 'temperature_2m', 'cloud_cover']
X = df[features]

# Define our targets (Labels)
rooms = ['occ_living_room', 'occ_kitchen', 'occ_master_bed']

# We will store our 3 trained models in this dictionary
trained_models = {}

print("Splitting data into 80% Training and 20% Testing...")
X_train, X_test, df_train, df_test = train_test_split(X, df, test_size=0.2, random_state=42)

Splitting data into 80% Training and 20% Testing...


In [13]:
for room in rooms:
    print(f"\n🧠 Training XGBoost for: {room}...")
    
    y_train = df_train[room]
    y_test = df_test[room]
    
    # Initialize the XGBoost Classifier
    model = xgb.XGBClassifier(
        n_estimators=100, 
        max_depth=4, 
        learning_rate=0.1, 
        use_label_encoder=False, 
        eval_metric='logloss'
    )
    
    # Train it!
    model.fit(X_train, y_train)
    
    # Test it!
    predictions = model.predict(X_test)
    acc = accuracy_score(y_test, predictions)
    
    print(f"✅ Accuracy for {room}: {acc * 100:.2f}%")
    
    # Save the trained model to our dictionary
    trained_models[room] = model


🧠 Training XGBoost for: occ_living_room...


d:\python project 1\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:47:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Accuracy for occ_living_room: 87.76%

🧠 Training XGBoost for: occ_kitchen...


d:\python project 1\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:47:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Accuracy for occ_kitchen: 81.52%

🧠 Training XGBoost for: occ_master_bed...
✅ Accuracy for occ_master_bed: 94.23%


d:\python project 1\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:47:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [14]:
# Save the dictionary of models to the backend store
model_path = "../backend/model_store/xgboost_occupancy_v1.pkl"
joblib.dump(trained_models, model_path)

print(f"✅ All 3 XGBoost models serialized and saved to: {model_path}")

✅ All 3 XGBoost models serialized and saved to: ../backend/model_store/xgboost_occupancy_v1.pkl


In [15]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')